# VehicleMPC 控制器使用样例

演示基于 CasADi 的 MPC 控制器：
1. 直线路径跟踪
2. 弯道路径跟踪
3. 与 RoadVehicleEnv 环境集成
4. 不同 MPC 参数对比

需要: `pip install casadi`

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from sim_env.vehicle_model import VehicleModel, VehicleModelConfig, VehicleParams, ModelType, IntegratorType
from sim_env.vehicle_controller import VehicleMPC, MPCConfig
from sim_env.road_model import RoadModel, RoadGenerationConfig, RoadSegmentType, SegmentSpec

plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
%matplotlib inline

In [ ]:
def draw_vehicle_box(ax, x, y, theta, color="lime", alpha=0.4, label=None,
                       length=None, width=None):
    """在 ax 上画车辆 BOX，尺寸默认取自 VehicleParams。"""
    _default = VehicleParams()
    L = length if length is not None else _default.length
    W = width if width is not None else _default.width
    hl, hw = L / 2, W / 2
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    corners = [(x + lx*cos_t - ly*sin_t, y + lx*sin_t + ly*cos_t)
               for lx, ly in [(-hl,-hw),(hl,-hw),(hl,hw),(-hl,hw)]]
    corners.append(corners[0])
    cx, cy = zip(*corners)
    ax.fill(cx, cy, color=color, alpha=alpha, zorder=4)
    ax.plot(cx, cy, color=color, lw=1.5, zorder=5, label=label)
    # 朝向箭头
    ax.annotate("", xy=(x + hl*1.3*cos_t, y + hl*1.3*sin_t), xytext=(x, y),
                arrowprops=dict(arrowstyle="->", color=color, lw=1.5), zorder=6)

## 示例 1：直线路径跟踪

MPC 控制车辆从静止加速，跟踪一条直线参考路径。

In [ ]:
# 参考路径：沿 x 轴的直线
ref_path = np.column_stack([np.linspace(0, 100, 50), np.zeros(50)])

# 车辆模型 + MPC 控制器
vp = VehicleParams()
vm = VehicleModel(cfg=VehicleModelConfig(vehicle_params=vp, model_type=ModelType.KINEMATIC, integrator_type=IntegratorType.RK4))
mpc = VehicleMPC(mpc_config=MPCConfig(vehicle_params=vp, horizon=15, target_speed=10.0))

vm.reset(x=0, y=4.0, theta=0.1, v=0.0)  # 初始有横向偏移和朝向偏差
mpc.reset()

DT = 0.1
xs, ys, thetas, vs, accels, omegas = [], [], [], [], [], []

for step in range(200):
    s = vm.state
    state = np.array([s.x, s.y, s.theta, s.v, s.steering])
    action, pred_xy, _ = mpc.compute(state, ref_path)
    vm.step(action, DT)

    xs.append(s.x)
    ys.append(s.y)
    thetas.append(s.theta)
    vs.append(s.v)
    accels.append(action[0])
    omegas.append(action[1])

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

ax = axes[0, 0]
ax.plot(ref_path[:, 0], ref_path[:, 1], "r--", lw=2, label="参考路径")
ax.plot(xs, ys, "b-", lw=1.5, label="实际轨迹")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("轨迹跟踪")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect("equal")

ax = axes[0, 1]
ax.plot(vs, "tab:orange")
ax.axhline(10.0, color="r", ls="--", label="目标速度")
ax.set_xlabel("步数")
ax.set_ylabel("速度 (m/s)")
ax.set_title("速度曲线")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.plot(accels, "tab:green")
ax.set_xlabel("步数")
ax.set_ylabel("加速度 (m/s²)")
ax.set_title("加速度")
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.plot(omegas, "tab:purple")
ax.set_xlabel("步数")
ax.set_ylabel("转向速度 (rad/s)")
ax.set_title("转向速度")
ax.grid(True, alpha=0.3)

# 画车辆 BOX（每隔 40 步画一个）
ax = axes[0, 0]
for idx in range(0, len(xs), 40):
    draw_vehicle_box(ax, xs[idx], ys[idx], thetas[idx],
                     color="lime", alpha=0.3, label="车辆" if idx == 0 else None)
ax.legend()

fig.suptitle("示例 1：直线路径跟踪（初始偏移 4m）", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## 示例 2：弯道路径跟踪

使用 RoadModel 生成弯道，MPC 跟踪中心线。展示预测轨迹。

In [ ]:
# 生成弯道
road = RoadModel(RoadGenerationConfig(num_lanes=2, lane_width=3.5))
geo = road.generate_fixed([
    SegmentSpec(RoadSegmentType.STRAIGHT, {"length": 30}),
    SegmentSpec(RoadSegmentType.CURVE, {"radius": 10, "angle_deg": 180}),
    SegmentSpec(RoadSegmentType.STRAIGHT, {"length": 30}),
])

vp = VehicleParams()
vm = VehicleModel(cfg=VehicleModelConfig(vehicle_params=vp, model_type=ModelType.KINEMATIC, integrator_type=IntegratorType.RK4))
mpc = VehicleMPC(mpc_config=MPCConfig(vehicle_params=vp, horizon=10, target_speed=8.0, dt=0.1))

vm.reset(x=0, y=0, theta=0, v=5.0)
mpc.reset()

DT = 0.1
xs, ys, thetas, preds = [], [], [], []

for step in range(300):
    s = vm.state
    state = np.array([s.x, s.y, s.theta, s.v, s.steering])

    # 获取前方参考路径
    cl, lb_pts, rb_pts, div_pts, nearest_idx, actual_num_points, actual_length_m, road_types = road.get_road_segment_ahead(s.x, s.y, num_points=25, pad=True)
    action, pred_xy, _ = mpc.compute(state, cl)
    vm.step(action, DT)

    xs.append(s.x)
    ys.append(s.y)
    thetas.append(s.theta)
    if step % 30 == 0:
        preds.append(pred_xy.copy())

fig, ax = plt.subplots(figsize=(10, 10))

# 道路
for seg in geo.road_segments:
    ax.plot(seg.centerline[:, 0], seg.centerline[:, 1], "y--", lw=1)
    ax.plot(seg.left_boundary[:, 0], seg.left_boundary[:, 1], "gray", lw=2)
    ax.plot(seg.right_boundary[:, 0], seg.right_boundary[:, 1], "gray", lw=2)

# 实际轨迹
ax.plot(xs, ys, "b-", lw=2, label="实际轨迹")
ax.plot(xs[0], ys[0], "go", ms=10)
ax.plot(xs[-1], ys[-1], "rs", ms=10)

# 预测轨迹（每隔一段画一次）
for i, pred in enumerate(preds):
    label = "MPC 预测" if i == 0 else None
    ax.plot(pred[:, 0], pred[:, 1], "c-", lw=1, alpha=0.5, label=label)
    ax.plot(pred[0, 0], pred[0, 1], "c.", ms=4)

# 画车辆 BOX（每隔 50 步）
for idx in range(0, len(xs), 50):
    draw_vehicle_box(ax, xs[idx], ys[idx], thetas[idx],
                     color="lime", alpha=0.3, label="车辆" if idx == 0 else None)

ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("示例 2：弯道路径跟踪 + MPC 预测轨迹")
ax.set_aspect("equal")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 示例 3：与 RoadVehicleEnv 集成

MPC 作为策略驱动 gymnasium 环境，从 obs 中提取状态和参考路径。

In [ ]:
from sim_env import RoadVehicleEnv, EnvConfig

env = RoadVehicleEnv(EnvConfig(
    road_config=RoadGenerationConfig(
        num_lanes=1,
        fixed_segments=[
            SegmentSpec(RoadSegmentType.STRAIGHT, {"length": 50}),
            SegmentSpec(RoadSegmentType.CURVE, {"radius": 35, "angle_deg": 70}),
            SegmentSpec(RoadSegmentType.STRAIGHT, {"length": 40}),
        ],
    ),
    road_points_ahead=25,
    init_speed=5.0,
    dt=0.1,
))

mpc = VehicleMPC(mpc_config=MPCConfig(vehicle_params=env._vehicle.params, horizon=20, target_speed=10.0, dt=0.1),
)

obs, info = env.reset(seed=0)
mpc.reset()

trajectory = {"x": [], "y": [], "v": [], "reward": [], "steering": []}
total_reward = 0

for step in range(500):
    # 从 obs 提取 MPC 输入
    veh = obs["vehicle"]
    state = np.array([veh[0], veh[1], veh[2], veh[3], info.get("steering", 0.0)])
    ref_path = obs["centerline"]  # (N, 2)

    action, _, _ = mpc.compute(state, ref_path)
    obs, reward, terminated, truncated, info = env.step(action)

    total_reward += reward
    trajectory["x"].append(info["x"])
    trajectory["y"].append(info["y"])
    trajectory["v"].append(info["v"])
    trajectory["reward"].append(reward)
    trajectory["steering"].append(info.get("steering", 0.0))

    if terminated or truncated:
        break

env.close()
print(f"Episode: steps={info['step']}, total_reward={total_reward:.1f}")
print(f"terminated={terminated}, truncated={truncated}")

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

ax = axes[0, 0]
ax.plot(trajectory["x"], trajectory["y"], "b-", lw=1.5, label="MPC 轨迹")
ax.plot(trajectory["x"][0], trajectory["y"][0], "go", ms=10)
ax.plot(trajectory["x"][-1], trajectory["y"][-1], "rs", ms=10)
ax.set_title("轨迹")
ax.set_aspect("equal")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(trajectory["v"], "tab:orange")
ax.axhline(10.0, color="r", ls="--", alpha=0.5)
ax.set_title("速度")
ax.set_ylabel("m/s")
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.plot(np.cumsum(trajectory["reward"]), "tab:green")
ax.set_title("累计奖励")
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.plot(np.degrees(trajectory["steering"]), "tab:purple")
ax.set_title("转向角")
ax.set_ylabel("度")
ax.grid(True, alpha=0.3)

# 画车辆 BOX
ax = axes[0, 0]
thetas_3 = [np.arctan2(trajectory["y"][min(j+1,len(trajectory["y"])-1)]-trajectory["y"][j],
            trajectory["x"][min(j+1,len(trajectory["x"])-1)]-trajectory["x"][j])
            for j in range(len(trajectory["x"]))]
for idx in range(0, len(trajectory["x"]), 80):
    draw_vehicle_box(ax, trajectory["x"][idx], trajectory["y"][idx], thetas_3[idx],
                     color="lime", alpha=0.3, label="车辆" if idx == 0 else None)
ax.legend()

fig.suptitle("示例 3：MPC + RoadVehicleEnv", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## 示例 4：不同 MPC 参数对比

对比不同预测步长和权重配置对跟踪效果的影响。

In [ ]:
# 生成测试道路
road = RoadModel(RoadGenerationConfig(num_lanes=1))
geo = road.generate_fixed([
    SegmentSpec(RoadSegmentType.STRAIGHT, {"length": 30}),
    SegmentSpec(RoadSegmentType.CURVE, {"radius": 25, "angle_deg": 90}),
    SegmentSpec(RoadSegmentType.STRAIGHT, {"length": 30}),
])

configs = [
    (MPCConfig(horizon=10, target_speed=5.0), "N=10", "tab:blue"),
    (MPCConfig(horizon=20, target_speed=5.0), "N=20", "tab:orange"),
    (MPCConfig(horizon=20, target_speed=5.0, w_pos=50.0), "N=20, w_pos=50", "tab:green"),
    (MPCConfig(horizon=20, target_speed=5.0, w_omega=5.0), "N=20, w_ω=5", "tab:red"),
]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for cfg, label, color in configs:
    vp = VehicleParams()
    vm = VehicleModel(cfg=VehicleModelConfig(vehicle_params=vp, model_type=ModelType.KINEMATIC, integrator_type=IntegratorType.RK4))
    mpc = VehicleMPC(mpc_config=cfg)
    vm.reset(x=0, y=0, theta=0, v=3.0)
    mpc.reset()

    xs, ys, vs = [], [], []
    for _ in range(250):
        s = vm.state
        state = np.array([s.x, s.y, s.theta, s.v, s.steering])
        cl, lb_pts, rb_pts, div_pts, nearest_idx, actual_num_points, actual_length_m, road_types = road.get_road_segment_ahead(s.x, s.y, num_points=25, pad=True)
        action, _, _ = mpc.compute(state, cl)
        vm.step(action, 0.1)
        xs.append(s.x)
        ys.append(s.y)
        vs.append(s.v)

    axes[0].plot(xs, ys, color=color, lw=2, label=label)
    axes[1].plot(vs, color=color, lw=1.5, label=label)

# 道路
for seg in geo.road_segments:
    axes[0].plot(seg.centerline[:, 0], seg.centerline[:, 1], "k--", lw=1, alpha=0.5)
    axes[0].plot(seg.left_boundary[:, 0], seg.left_boundary[:, 1], "gray", lw=1)
    axes[0].plot(seg.right_boundary[:, 0], seg.right_boundary[:, 1], "gray", lw=1)
axes[0].set_xlabel("x (m)")
axes[0].set_ylabel("y (m)")
axes[0].set_title("轨迹对比")
axes[0].set_aspect("equal")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].axhline(8.0, color="k", ls="--", alpha=0.3, label="目标速度")
axes[1].set_xlabel("步数")
axes[1].set_ylabel("速度 (m/s)")
axes[1].set_title("速度对比")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

fig.suptitle("示例 4：不同 MPC 参数对比", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()